# 02 - Adversarial eps-Curves Overlay

Plot accuracy-vs-epsilon curves for all runs in a sweep, color-coded by varying dial.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from playground.runs_io import load_sweep

In [ ]:
EXPERIMENT = "smoke_sweep"
runs_dir = Path.cwd().parent / "runs" / EXPERIMENT
if not runs_dir.exists():
    print(f"No runs at {runs_dir}.")
    df = pd.DataFrame()
else:
    df = load_sweep(runs_dir)
    print(f"loaded {len(df)} runs")

In [ ]:
if not df.empty:
    probe_prefixes = ("margin_", "grad_", "adversarial_", "calibration_", "thickness_", "hessian_", "landscape_")
    summary_suffixes = ("_final", "_best")
    skip_cols = {"run_dir", "experiment_name", "run_id", "extra", "description"}
    config_cols = [c for c in df.columns if not c.endswith(summary_suffixes) and not c.startswith(probe_prefixes) and c not in skip_cols]
    varying = [c for c in config_cols if df[c].nunique() > 1 and c != "seed"]

    if varying:
        dial = varying[0]
        fig, ax = plt.subplots(figsize=(7, 4))
        for _, row in df.iterrows():
            adv_path = Path(row["run_dir"]) / "probes" / "adversarial.json"
            if not adv_path.exists():
                continue
            adv = json.loads(adv_path.read_text())
            ax.plot(adv["epsilons"], adv["accuracy_by_eps"], marker="o",
                    label=f"{dial}={row[dial]}, seed={row['seed']}", alpha=0.7)
        ax.set_xlabel("eps")
        ax.set_ylabel("accuracy")
        ax.set_title(f"Adversarial accuracy vs eps, colored by {dial}")
        ax.legend(fontsize=7)
        plt.show()